[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/068.%20The%20AS_RS%20Task%20Interleaving%20Problem/P68-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 68. The AS/RS Task Interleaving Problem
## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

## Purpose and scope

An automated storage and retrieval system (AS/RS) uses a crane to carry out storage requests and retrieval requests. When the crane completes one task before starting the next, the order of tasks determines how far it travels. **Task interleaving** means choosing one combined order for both types of request rather than processing all storage tasks first and all retrieval tasks afterward.

This notebook models one crane, a fixed set of tasks, a depot, task service times, and a specified travel-time metric. It finds a minimum-time sequence in two ways: exhaustive enumeration, which provides an exact benchmark for small instances, and a mixed-integer programming (MIP) model that represents task positions and consecutive transitions. Both methods minimize the same complete-cycle-time objective.

### Model assumptions

- A location is represented by integer coordinates $(x,y)$.
- The crane's travel time between locations is defined by the maximum coordinate displacement:
  $$
  t(p,q)=0.5\max\left(|x_q-x_p|,|y_q-y_p|\right)\text{ seconds},
  $$
  where $p=(x_p,y_p)$ and $q=(x_q,y_q)$. This is a maximum-coordinate, or Chebyshev, metric rather than the Manhattan metric. It represents a crane that can move along both axes simultaneously at equal speed; one coordinate unit of maximum displacement takes $0.5$ seconds.
- Each storage task takes 3 seconds to perform, and each retrieval task takes 2 seconds.
- Every task is performed exactly once, and the crane performs one task at a time.
- The crane starts at the depot and returns there after the final task.
- Task identifiers beginning with `S` denote storage tasks; identifiers beginning with `R` denote retrieval tasks. The task records also contain priorities, but priorities do not affect the objective. There are no due-time penalties in this model.
- Travel and service times are additive. The model does not include travel restrictions, acceleration, congestion, blocking, or uncertainty.

### Solution sequence

1. Define the task data, depot, travel-time metric, and service times.
2. Define the total time of a complete task sequence.
3. Enumerate all task orders to obtain an exact small-instance benchmark.
4. Formulate the same sequencing objective as a position-assignment MIP with variables for consecutive task transitions.
5. Compare the sequential order with the best interleaved order, then examine depot-location sensitivity and instance size.

### How to interpret the results

A sequence is a complete order of task identifiers. Its total time includes travel to the first task, travel between consecutive tasks, service time at every task, and the return trip to the depot. Because each task is served exactly once, total service time is constant across all feasible sequences. Thus, differences in total cycle time are due to travel, although the reported percentage reduction compares total cycle times. Several distinct sequences can attain the same minimum; each solver reports one minimum-time sequence.

## Mathematical model

Let $N=\{1,\ldots,n\}$ be the set of tasks, and let $p_i$ and $s_i$ denote the location and service time of task $i$. Let $p_0$ be the depot. The travel time between tasks $i$ and $k$ is $c_{ik}=t(p_i,p_k)$; depot-to-task and task-to-depot costs are defined in the same way. In the code, tasks are stored in `all_tasks`, the depot in `depot`, and travel costs are calculated by `calculate_travel_time`.

### Exact sequence objective

A sequence is a permutation $\pi=(\pi_1,\ldots,\pi_n)$ of the task set. Its total cycle time is
$$
T(\pi)=c_{0,\pi_1}+\sum_{r=1}^{n}s_{\pi_r}+\sum_{r=1}^{n-1}c_{\pi_r,\pi_{r+1}}+c_{\pi_n,0}.
$$
Here, $c_{0,\pi_1}$ is travel from the depot to the first task, $s_{\pi_r}$ is the service time of the task in position $r$, $c_{\pi_r,\pi_{r+1}}$ is travel between consecutive tasks, and $c_{\pi_n,0}$ is the return to the depot. The exhaustive method selects a permutation minimizing $T(\pi)$. It examines $n!$ permutations, and evaluating each sequence takes $O(n)$ time, so its running time is $O(n\,n!)$. The `itertools.permutations` iterator generates permutations as needed instead of storing the entire list. The code implements the objective in `calculate_sequence_time` and searches the permutations in `solve_brute_force`.

### Position-assignment MIP

The MIP makes the order explicit through binary decision variables. Let $x_{ij}=1$ when task $i$ occupies position $j$, and $x_{ij}=0$ otherwise. Let $y_{ikj}=1$ when task $i$ in position $j$ is immediately followed by task $k$ in position $j+1$, and $y_{ikj}=0$ otherwise. The objective includes travel from the depot, travel between consecutive tasks, service time, and travel back to the depot:
$$
\min\quad
\sum_{i\in N}c_{0i}x_{i1}
+\sum_{j=1}^{n-1}\sum_{i\in N}\sum_{\substack{k\in N\\k\ne i}}c_{ik}y_{ikj}
+\sum_{i\in N}s_i
+\sum_{i\in N}c_{i0}x_{in}.
$$
The first term charges travel from the depot to whichever task occupies the first position. The second charges travel for each selected consecutive-task transition. The third is the service time of every task, incurred once because every task must be assigned. The final term charges travel from the last task back to the depot.

The task-position assignment constraints are
$$
\sum_{j=1}^{n}x_{ij}=1\quad(i\in N),
\qquad
\sum_{i\in N}x_{ij}=1\quad(j=1,\ldots,n).
$$
The first equality places every task in exactly one position; the second assigns exactly one task to each position. For every pair of consecutive positions, transition variables are linked to the selected tasks by
$$
\sum_{\substack{k\in N\\k\ne i}}y_{ikj}=x_{ij}
\quad(i\in N,\ j=1,\ldots,n-1),
$$
$$
\sum_{\substack{i\in N\\i\ne k}}y_{ikj}=x_{k,j+1}
\quad(k\in N,\ j=1,\ldots,n-1),
$$
$$
x_{ij}\in\{0,1\},\qquad y_{ikj}\in\{0,1\}.
$$
The first transition equality ensures that a task selected at position $j$ has exactly one successor at position $j+1$. The second ensures that the task selected at position $j+1$ has exactly one predecessor. Since each position contains one task, these relationships select the actual transition between each adjacent pair of positions. The restriction $k\ne i$ is consistent with a permutation: a task cannot occupy two positions.

In the code, `x` represents the position variables and `travel_vars` represents the transition variables. The nested loops create the variables and constraints; `model.solve` asks the MIP solver to optimize the resulting model. The model has $n^2$ position variables and $(n-1)n(n-1)$ transition variables when all distinct-task transitions are represented. The formulation therefore grows polynomially in the number of variables, but solving a MIP can still become computationally difficult as instances grow. The MIP and exhaustive enumeration minimize the same complete-cycle-time objective, so comparing their objective values on small instances is a useful consistency check.

### Example instance

The example contains four storage requests, S1–S4, at $(2,3)$, $(6,5)$, $(8,2)$, and $(4,6)$, and three retrieval requests, R1–R3, at $(3,4)$, $(7,3)$, and $(5,7)$. The depot is $(1,1)$. The cells below calculate and report sequence times directly from these data and the stated model; priorities do not enter the objective.

In [ ]:
import math
from itertools import permutations

import matplotlib.pyplot as plt
import numpy as np
import pulp

plt.style.use("default")

In [ ]:
class ASRSMathematicalFormulation:
    """Solve the AS/RS task-interleaving problem by enumeration and MIP."""

    def __init__(self, storage_tasks, retrieval_tasks, depot=(1, 1)):
        """Store the task data and depot used by the sequencing model.

        Each task is a tuple (identifier, x, y, priority). Priority is retained
        as task metadata; it is not part of the objective in this model.
        """
        self.storage_tasks = storage_tasks
        self.retrieval_tasks = retrieval_tasks
        self.all_tasks = storage_tasks + retrieval_tasks
        self.depot = depot
        self.n_tasks = len(self.all_tasks)

    def calculate_travel_time(self, pos1, pos2):
        """Return travel time under the maximum-coordinate movement metric."""
        return max(abs(pos2[0] - pos1[0]), abs(pos2[1] - pos1[1])) * 0.5

    def calculate_sequence_time(self, sequence):
        """Calculate travel, service, and depot-return time for a sequence."""
        total_time = 0.0
        current_pos = self.depot

        for task_idx in sequence:
            task = self.all_tasks[task_idx]
            task_pos = (task[1], task[2])
            total_time += self.calculate_travel_time(current_pos, task_pos)
            total_time += 3 if task[0].startswith("S") else 2
            current_pos = task_pos

        total_time += self.calculate_travel_time(current_pos, self.depot)
        return total_time

    def solve_brute_force(self):
        """Return one minimum-time sequence and its time by full enumeration."""
        best_sequence = None
        best_time = float("inf")

        for sequence in permutations(range(self.n_tasks)):
            sequence_time = self.calculate_sequence_time(sequence)
            if sequence_time < best_time:
                best_time = sequence_time
                best_sequence = sequence

        return best_sequence, best_time

    def solve_mip(self):
        """Solve the position-assignment and consecutive-transition MIP."""
        if self.n_tasks == 0:
            return [], 0.0, 0.0

        model = pulp.LpProblem("AS_RS_Task_Interleaving", pulp.LpMinimize)

        # Positional arguments keep the binary category explicit across PuLP APIs.
        x = {
            (i, j): pulp.LpVariable(f"x_{i}_{j}", 0, 1, "Binary")
            for i in range(self.n_tasks)
            for j in range(self.n_tasks)
        }

        # Transition variables charge travel only for the selected adjacent pair.
        travel_vars = {}
        for j in range(self.n_tasks - 1):
            for i in range(self.n_tasks):
                for k in range(self.n_tasks):
                    if i != k:
                        travel_vars[(i, k, j)] = pulp.LpVariable(
                            f"t_{i}_{k}_{j}", 0, 1, "Binary"
                        )

        objective_terms = []

        # Only the first and last task use depot travel costs.
        for i, task in enumerate(self.all_tasks):
            task_pos = (task[1], task[2])
            objective_terms.append(
                self.calculate_travel_time(self.depot, task_pos) * x[(i, 0)]
            )
            objective_terms.append(
                self.calculate_travel_time(task_pos, self.depot)
                * x[(i, self.n_tasks - 1)]
            )

        # Each task's service time is incurred once in every feasible route.
        for i, task in enumerate(self.all_tasks):
            service_time = 3 if task[0].startswith("S") else 2
            objective_terms.append(service_time)

        for (i, k, j), variable in travel_vars.items():
            task_i = self.all_tasks[i]
            task_k = self.all_tasks[k]
            pos_i = (task_i[1], task_i[2])
            pos_k = (task_k[1], task_k[2])
            travel_time = self.calculate_travel_time(pos_i, pos_k)
            objective_terms.append(travel_time * variable)

        model += pulp.lpSum(objective_terms)

        # Each task appears once, and each position is occupied once.
        for i in range(self.n_tasks):
            model += pulp.lpSum(
                x[(i, j)] for j in range(self.n_tasks)
            ) == 1

        for j in range(self.n_tasks):
            model += pulp.lpSum(
                x[(i, j)] for i in range(self.n_tasks)
            ) == 1

        # Link each selected task to one successor and each next task to one predecessor.
        for j in range(self.n_tasks - 1):
            for i in range(self.n_tasks):
                model += pulp.lpSum(
                    travel_vars[(i, k, j)]
                    for k in range(self.n_tasks)
                    if k != i
                ) == x[(i, j)]

            for k in range(self.n_tasks):
                model += pulp.lpSum(
                    travel_vars[(i, k, j)]
                    for i in range(self.n_tasks)
                    if i != k
                ) == x[(k, j + 1)]

        model.solve(pulp.PULP_CBC_CMD(msg=False))

        if model.status != pulp.LpStatusOptimal:
            return None, None, None

        sequence = []
        for j in range(self.n_tasks):
            selected_tasks = [
                i for i in range(self.n_tasks)
                if pulp.value(x[(i, j)]) is not None
                and pulp.value(x[(i, j)]) > 0.5
            ]
            if len(selected_tasks) != 1:
                return None, None, None
            sequence.append(selected_tasks[0])

        total_time = self.calculate_sequence_time(sequence)
        objective_value = pulp.value(model.objective)
        return sequence, total_time, objective_value

In [ ]:
# The example data include priority metadata, which the stated objective does not use.
storage_tasks = [
    ("S1", 2, 3, 5),
    ("S2", 6, 5, 3),
    ("S3", 8, 2, 4),
    ("S4", 4, 6, 2),
]

retrieval_tasks = [
    ("R1", 3, 4, 8),
    ("R2", 7, 3, 6),
    ("R3", 5, 7, 2),
]

asrs_solver = ASRSMathematicalFormulation(storage_tasks, retrieval_tasks)

print("AS/RS Task Interleaving Problem — Mathematical Formulation")
print(f"Storage tasks: {len(storage_tasks)}")
print(f"Retrieval tasks: {len(retrieval_tasks)}")
print(f"Total tasks: {asrs_solver.n_tasks}")
print(f"Depot position: {asrs_solver.depot}")
print("\nTask details:")
for i, task in enumerate(asrs_solver.all_tasks):
    print(
        f"  {i}: {task[0]} at ({task[1]}, {task[2]}), "
        f"priority metadata={task[3]}"
    )

In [ ]:
print("=== Exhaustive Enumeration (Exact for This Instance) ===")
optimal_sequence, optimal_time = asrs_solver.solve_brute_force()

if optimal_sequence is not None:
    optimal_task_names = [
        asrs_solver.all_tasks[i][0] for i in optimal_sequence
    ]
    print(f"One minimum-time sequence: {optimal_task_names}")
    print(f"Minimum total cycle time: {optimal_time:.2f} seconds")

    sequential_sequence = list(range(len(storage_tasks))) + list(
        range(len(storage_tasks), asrs_solver.n_tasks)
    )
    sequential_time = asrs_solver.calculate_sequence_time(sequential_sequence)
    print(f"Sequential cycle time: {sequential_time:.2f} seconds")

    savings = (sequential_time - optimal_time) / sequential_time * 100
    print(f"Cycle-time reduction: {savings:.1f}%")
else:
    print("No feasible sequence was found.")

In [ ]:
print("\n=== Mixed-Integer Programming Solution ===")
mip_sequence, mip_time, objective_value = asrs_solver.solve_mip()

if mip_sequence is not None:
    mip_task_names = [asrs_solver.all_tasks[i][0] for i in mip_sequence]
    print(f"MIP sequence: {mip_task_names}")
    print(f"MIP total cycle time: {mip_time:.2f} seconds")
    print(f"MIP objective value: {objective_value:.2f} seconds")
    assert math.isclose(mip_time, objective_value, rel_tol=1e-9, abs_tol=1e-9)

    if optimal_sequence is not None:
        gap = (mip_time - optimal_time) / optimal_time * 100
        print(f"Difference from enumerated optimum: {gap:.2f}%")
        assert math.isclose(mip_time, optimal_time, rel_tol=1e-9, abs_tol=1e-9)
else:
    print("The MIP solver did not report an optimal solution.")

In [ ]:
print("\n=== Sensitivity Analysis: Depot Position ===")

depot_positions = [(1, 1), (0, 0), (5, 5), (10, 1)]
results = []

for depot_pos in depot_positions:
    solver_temp = ASRSMathematicalFormulation(
        storage_tasks, retrieval_tasks, depot=depot_pos
    )
    sequence, cycle_time = solver_temp.solve_brute_force()
    results.append((depot_pos, cycle_time))
    print(f"Depot {depot_pos}: minimum cycle time = {cycle_time:.2f} seconds")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

depot_x = [position[0] for position, _ in results]
depot_y = [position[1] for position, _ in results]
cycle_times = [cycle_time for _, cycle_time in results]
scatter = ax1.scatter(
    depot_x, depot_y, c=cycle_times, s=100, cmap="viridis"
)
ax1.set_xlabel("Depot x-coordinate")
ax1.set_ylabel("Depot y-coordinate")
ax1.set_title("Depot Location and Minimum Cycle Time")
ax1.grid(True, alpha=0.3)
fig.colorbar(scatter, ax=ax1, label="Minimum cycle time (seconds)")

# The line connects task locations in service order; it illustrates the
# sequence, not the crane's detailed physical path between rack locations.
for task in asrs_solver.all_tasks:
    is_storage = task[0].startswith("S")
    color = "red" if is_storage else "blue"
    label = "Storage task" if is_storage else "Retrieval task"
    if label not in ax2.get_legend_handles_labels()[1]:
        ax2.scatter(task[1], task[2], c=color, s=100, alpha=0.75, label=label)
    else:
        ax2.scatter(task[1], task[2], c=color, s=100, alpha=0.75)
    ax2.annotate(task[0], (task[1], task[2]), xytext=(5, 5), textcoords="offset points")

ax2.scatter(
    asrs_solver.depot[0], asrs_solver.depot[1],
    c="green", s=200, marker="s", label="Depot"
)

if optimal_sequence is not None:
    route_x = [asrs_solver.depot[0]]
    route_y = [asrs_solver.depot[1]]
    for task_idx in optimal_sequence:
        task = asrs_solver.all_tasks[task_idx]
        route_x.append(task[1])
        route_y.append(task[2])
    route_x.append(asrs_solver.depot[0])
    route_y.append(asrs_solver.depot[1])
    ax2.plot(
        route_x, route_y, "g--", alpha=0.6, linewidth=2,
        label="Task order (sequence connections)"
    )

ax2.set_xlabel("x-coordinate")
ax2.set_ylabel("y-coordinate")
ax2.set_title("Minimum-Time Task Sequence")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Sequential and Interleaved Cycle Times by Instance Size ===")

# Prefix subsets use only the task records supplied in the example.
problem_sizes = [4, 6, 7]
sequential_times = []
optimal_times = []
savings_percentages = []

for size in problem_sizes:
    n_storage = size // 2
    n_retrieval = size - n_storage
    storage_sample = storage_tasks[:n_storage]
    retrieval_sample = retrieval_tasks[:n_retrieval]

    solver = ASRSMathematicalFormulation(storage_sample, retrieval_sample)
    sequential_sequence = list(range(n_storage)) + list(
        range(n_storage, size)
    )
    sequential_time = solver.calculate_sequence_time(sequential_sequence)
    _, minimum_time = solver.solve_brute_force()

    savings = (sequential_time - minimum_time) / sequential_time * 100
    sequential_times.append(sequential_time)
    optimal_times.append(minimum_time)
    savings_percentages.append(savings)
    print(
        f"Size {size}: sequential={sequential_time:.2f} s, "
        f"minimum={minimum_time:.2f} s, cycle-time reduction={savings:.1f}%"
    )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
positions = np.arange(len(problem_sizes))
width = 0.35

ax1.bar(
    positions - width / 2, sequential_times, width,
    label="Sequential", alpha=0.75
)
ax1.bar(
    positions + width / 2, optimal_times, width,
    label="Minimum-time interleaving", alpha=0.75
)
ax1.set_xlabel("Number of tasks")
ax1.set_ylabel("Total cycle time (seconds)")
ax1.set_title("Sequential and Minimum-Time Cycle Times")
ax1.set_xticks(positions)
ax1.set_xticklabels([f"{size} tasks" for size in problem_sizes])
ax1.legend()
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(positions, savings_percentages, color="green", alpha=0.7)
ax2.set_xlabel("Number of tasks")
ax2.set_ylabel("Cycle-time reduction (%)")
ax2.set_title("Reduction from Interleaving")
ax2.set_xticks(positions)
ax2.set_xticklabels([f"{size} tasks" for size in problem_sizes])
ax2.grid(True, axis="y", alpha=0.3)

for position, savings in zip(positions, savings_percentages):
    ax2.text(position, savings, f"{savings:.1f}%", ha="center", va="bottom")

plt.tight_layout()
plt.show()

average_savings = float(np.mean(savings_percentages))
print(f"Average cycle-time reduction across these sizes: {average_savings:.1f}%")

## Strengths, limitations, and appropriate use

### Strengths

- Exhaustive enumeration guarantees a globally minimum sequence for a finite instance when the enumeration completes.
- The position-assignment MIP expresses the same travel-and-service objective using explicit binary decisions and constraints.
- Comparing the two formulations provides a useful consistency check on small instances.
- Changing the depot location gives a direct sensitivity study while keeping task data and the objective fixed.

### Limitations

- Exhaustive enumeration evaluates $n!$ sequences, with $O(n\,n!)$ running time when each sequence is evaluated in $O(n)$ time. Its practical use is therefore limited to small instances.
- The MIP has $n^2$ position variables and $(n-1)n(n-1)$ transition variables for the distinct-task transitions represented here, in addition to assignment and linking constraints. MIP solution time depends on the instance and solver.
- The travel metric assumes simultaneous movement along the two coordinate axes at equal speed. It does not represent aisle restrictions, acceleration, congestion, blocking, or other physical operating details.
- Priorities, due times, multiple cranes, and task precedence are outside the stated model. Including them would require corresponding objective terms or constraints and is not part of this example.

### Interpretation

The sequential order is a reference policy: all storage requests are followed by all retrieval requests. Interleaving can reduce travel by placing tasks from both groups near one another in the service order. The reported cycle-time reduction is specific to the supplied coordinates, depot, travel metric, and service times; it is not a guarantee for every instance. The exact methods are most useful here for learning the formulation and studying small cases, where a complete sequence can be inspected and verified.